# Taller Spark: ETL local de ventas

Notebook resuelto y explicado. **Cómo usarlo:** ejecuta las celdas en orden con **Shift + Enter**.
Cada paso tiene primero una celda de texto que explica *qué* hacemos y *por qué*, luego el código
con comentarios línea por línea y, debajo, el resultado.

| Paso | Tema |
|---|---|
| 1 | Arrancar Spark |
| 2 | Leer los CSV |
| 3 | Perfilar: buscar problemas de calidad |
| 4 | Limpiar cada tabla |
| 5 | Unir tablas, clasificar líneas y calcular el importe neto |
| 6 | Agregar ventas por mes, país, canal y categoría |
| 7 | Guardar Parquet y controles de calidad |
| 8 | Incremental idempotente |

## Paso 1: Arrancar Spark

Spark es un motor para procesar datos en paralelo. Normalmente corre en un **clúster** (muchas
máquinas), pero aquí lo corremos en **modo local**: tu Mac hace de clúster.

La `SparkSession` es la "puerta de entrada": todo lo que hagas con Spark pasa por el objeto `spark`.

In [5]:
# Importamos las 3 piezas que usaremos en todo el notebook:
#  - SparkSession: para crear la sesión de Spark
#  - Window: para "funciones de ventana" (ej. numerar filas dentro de un grupo)
#  - functions as F: la caja de herramientas de funciones (col, when, sum, round...)
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder                              # el "constructor" de la sesión
    .master("local[*]")                               # local = en tu Mac; [*] = usa todos los núcleos de CPU
    .appName("taller-data-engineer")                  # nombre que aparece en los logs / Spark UI
    .config("spark.sql.session.timeZone", "UTC")      # las fechas vienen en UTC (terminan en 'Z'); evitamos que se corran a hora Colombia
    .getOrCreate()                                    # crea la sesión, o reutiliza la que ya exista
)
spark.sparkContext.setLogLevel("WARN")                # menos ruido en la salida: solo advertencias y errores

print("Spark versión:", spark.version)

Spark versión: 3.5.6


## Paso 2: Leer los CSV

Leemos los 4 archivos con `inferSchema=False`: **todas las columnas llegan como texto (string)**.
¿Por qué? Porque los datos están sucios. Si Spark adivinara los tipos, un precio `"-12.50"` o una
fecha `"2026-99-31"` podrían convertirse en algo raro sin avisarte. Leemos todo como texto y luego
**nosotros** decidimos cómo convertir y qué hacer con lo que no convierte.

**Granularidad (qué representa 1 fila):** customers = 1 cliente, products = 1 producto,
orders = 1 pedido, order_items = 1 línea de un pedido. Ojo: `order_items` no trae precio;
el precio está en `products`.

In [6]:
def leer_csv(nombre):
    # Función auxiliar para no repetir el mismo código 4 veces.
    return (
        spark.read                              # lector de archivos de Spark
        .option("header", True)                 # la primera fila son los nombres de columna
        .option("inferSchema", False)           # NO adivinar tipos: todo entra como string
        .csv(f"datos/{nombre}.csv")             # ruta relativa: ejecuta el notebook desde la carpeta del taller
    )

customers_raw = leer_csv("customers")           # sufijo _raw = dato crudo, tal cual vino
products_raw  = leer_csv("products")
orders_raw    = leer_csv("orders")
items_raw     = leer_csv("order_items")

# .count() es una ACCIÓN: obliga a Spark a leer de verdad el archivo y contar filas.
for nombre, df in [("customers", customers_raw), ("products", products_raw),
                   ("orders", orders_raw), ("order_items", items_raw)]:
    print(f"{nombre:12s} {df.count():>6} filas")

orders_raw.printSchema()                        # muestra columnas y tipos (todos string)
orders_raw.show(5, truncate=False)              # muestra 5 filas sin cortar el texto

customers      1001 filas
products        100 filas
orders        20001 filas
order_items   40026 filas
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- updated_at: string (nullable = true)

+--------+-----------+--------------------+---------+-------+--------------------+
|order_id|customer_id|order_ts            |status   |channel|updated_at          |
+--------+-----------+--------------------+---------+-------+--------------------+
|O00001  |C0515      |2026-01-08T22:54:00Z|delivered|web    |2026-01-08T22:54:00Z|
|O00002  |C0637      |2026-02-28T08:30:00Z|pending  |store  |2026-02-28T08:30:00Z|
|O00003  |C0770      |2026-02-06T04:45:00Z|pending  |web    |2026-02-06T04:45:00Z|
|O00004  |C0083      |2026-01-02T08:57:00Z|delivered|mobile |2026-01-02T08:57:00Z|
|O00005  |C0461      |2026-02-22T15:20:00Z|delivered|store  |2026-0

**Primera pista:** el enunciado dice 1.000 clientes y 20.000 pedidos, pero contamos **1.001** y **20.001**.
Algo está duplicado. Eso lo confirmamos en el paso 3.

> **Concepto: transformación vs acción.** `spark.read...csv()` todavía no lee nada: Spark es *perezoso* (lazy),
> solo anota el plan. Las **acciones** (`count`, `show`, `collect`, `write`) son las que disparan el trabajo real.

## Paso 3: Perfilar los datos (buscar problemas)

Revisamos 5 cosas:
1. **Claves duplicadas:** ¿la clave es realmente única?
2. **Nulos:** ¿qué columnas tienen valores vacíos?
3. **Dominios:** ¿los valores están dentro de lo esperado? (estados, cantidades, descuentos)
4. **Tipos inválidos:** ¿hay precios o fechas que no se pueden convertir?
5. **Huérfanos:** ¿hay referencias a clientes, pedidos o productos que no existen?

In [7]:
def perfil_claves(df, clave):
    total     = df.count()                                   # filas totales
    distintos = df.select(clave).distinct().count()          # valores únicos de la clave
    nulos     = df.filter(F.col(clave).isNull()).count()     # filas con la clave vacía
    # Si total > distintos, hay claves repetidas.
    print(f"{clave:12s} total={total:>6} distintos={distintos:>6} nulos={nulos}  -> duplicados={total - distintos}")

print("== 1. Claves ==")
perfil_claves(customers_raw, "customer_id")
perfil_claves(products_raw,  "product_id")
perfil_claves(orders_raw,    "order_id")
perfil_claves(items_raw,     "line_id")

# Buscamos QUÉ clave está repetida: agrupamos por la clave y nos quedamos con las que aparecen > 1 vez.
customers_raw.groupBy("customer_id").count().filter("count > 1").show()
orders_raw.groupBy("order_id").count().filter("count > 1").show()

== 1. Claves ==
customer_id  total=  1001 distintos=  1000 nulos=0  -> duplicados=1
product_id   total=   100 distintos=   100 nulos=0  -> duplicados=0
order_id     total= 20001 distintos= 20000 nulos=0  -> duplicados=1
line_id      total= 40026 distintos= 40026 nulos=0  -> duplicados=0
+-----------+-----+
|customer_id|count|
+-----------+-----+
|      C0020|    2|
+-----------+-----+

+--------+-----+
|order_id|count|
+--------+-----+
|  O00012|    2|
+--------+-----+



In [8]:
def perfil_nulos(df, nombre):
    # Para cada columna c: isNull() da True/False -> cast("int") lo vuelve 1/0 -> sum cuenta los nulos.
    # La "list comprehension" [... for c in df.columns] genera una expresión por columna.
    print(nombre)
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

print("== 2. Nulos por columna ==")
perfil_nulos(customers_raw, "customers")
perfil_nulos(products_raw,  "products")
perfil_nulos(orders_raw,    "orders")
perfil_nulos(items_raw,     "order_items")

# Detalle: ¿qué emails están mal? (vacío o sin '@')
customers_raw.filter(F.col("email").isNull() | ~F.col("email").contains("@")).show()

== 2. Nulos por columna ==
customers
+-----------+----------+---------+-----+-------+-----------+-------+
|customer_id|first_name|last_name|email|country|signup_date|segment|
+-----------+----------+---------+-----+-------+-----------+-------+
|          0|         0|        0|    1|      0|          0|      0|
+-----------+----------+---------+-----+-------+-----------+-------+

products
+----------+------------+--------+----------+------+
|product_id|product_name|category|unit_price|active|
+----------+------------+--------+----------+------+
|         0|           0|       0|         1|     0|
+----------+------------+--------+----------+------+

orders
+--------+-----------+--------+------+-------+----------+
|order_id|customer_id|order_ts|status|channel|updated_at|
+--------+-----------+--------+------+-------+----------+
|       0|          1|       0|     0|      0|         0|
+--------+-----------+--------+------+-------+----------+

order_items
+-------+--------+----------+---

In [9]:
print("== 3. Dominios ==")
orders_raw.groupBy("status").count().orderBy("status").show()        # ¿algún estado mal escrito?
items_raw.groupBy("quantity").count().orderBy("quantity").show()     # ¿cantidades <= 0?
items_raw.groupBy("discount_pct").count().orderBy("discount_pct").show()  # ¿descuentos fuera de 0..1?

== 3. Dominios ==
+---------+-----+
|   status|count|
+---------+-----+
|cancelled| 5008|
| deliverd|    1|
|delivered| 5086|
|  pending| 4976|
|  shipped| 4930|
+---------+-----+

+--------+-----+
|quantity|count|
+--------+-----+
|      -2|    1|
|       0|    1|
|       1| 8047|
|       2| 7888|
|       3| 7943|
|       4| 8035|
|       5| 8111|
+--------+-----+

+------------+-----+
|discount_pct|count|
+------------+-----+
|        0.00|10070|
|        0.05|10022|
|        0.10| 9998|
|        0.15| 9935|
|        1.20|    1|
+------------+-----+



In [10]:
print("== 4. Tipos inválidos ==")
# try_cast intenta convertir texto -> decimal. Si NO puede, devuelve null en vez de lanzar error.
precio = F.expr("try_cast(unit_price AS decimal(10,2))")
products_raw.filter(precio.isNull() | (precio <= 0)).show()           # precios vacíos o <= 0

# try_to_timestamp intenta convertir texto -> fecha-hora; si la fecha es imposible, devuelve null.
orders_raw.filter(F.expr("try_to_timestamp(order_ts)").isNull()).show(truncate=False)

== 4. Tipos inválidos ==
+----------+------------+-----------+----------+------+
|product_id|product_name|   category|unit_price|active|
+----------+------------+-----------+----------+------+
|      P088|  Product 88|      books|    -12.50|  true|
|      P089|  Product 89|electronics|      NULL|  true|
+----------+------------+-----------+----------+------+

+--------+-----------+--------------------+-------+-------+--------------------+
|order_id|customer_id|order_ts            |status |channel|updated_at          |
+--------+-----------+--------------------+-------+-------+--------------------+
|O00025  |C0478      |2026-99-31T12:00:00Z|pending|mobile |2026-02-15T04:34:00Z|
+--------+-----------+--------------------+-------+-------+--------------------+



In [7]:
print("== 5. Huérfanos ==")
# left_anti join = "dame las filas de la IZQUIERDA que NO tienen pareja en la derecha".
# Es la forma clásica de encontrar referencias rotas.
print("Pedidos cuyo cliente no existe:")
orders_raw.join(customers_raw, "customer_id", "left_anti").show()
print("Líneas cuyo pedido no existe:")
items_raw.join(orders_raw, "order_id", "left_anti").show()
print("Líneas cuyo producto no existe:")
items_raw.join(products_raw, "product_id", "left_anti").show()

== 5. Huérfanos ==
Pedidos cuyo cliente no existe:


+-----------+--------+--------------------+---------+-------+--------------------+
|customer_id|order_id|            order_ts|   status|channel|          updated_at|
+-----------+--------+--------------------+---------+-------+--------------------+
|      C9999|  O00007|2026-01-17T23:52:00Z|cancelled| mobile|2026-01-17T23:52:00Z|
|       NULL|  O00011|2026-02-25T00:35:00Z|  pending| mobile|2026-02-25T00:35:00Z|
+-----------+--------+--------------------+---------+-------+--------------------+

Líneas cuyo pedido no existe:


+--------+--------+----------+--------+------------+
|order_id| line_id|product_id|quantity|discount_pct|
+--------+--------+----------+--------+------------+
|  O99999|L0000003|      P028|       3|        0.00|
+--------+--------+----------+--------+------------+

Líneas cuyo producto no existe:
+----------+--------+--------+--------+------------+
|product_id| line_id|order_id|quantity|discount_pct|
+----------+--------+--------+--------+------------+
|      P999|L0000004|  O00003|       2|        0.10|
+----------+--------+--------+--------+------------+



### Anomalías encontradas y decisión

| # | Tabla | Problema | Decisión | Por qué |
|---|---|---|---|---|
| 1 | customers | `C0020` duplicado (fila idéntica) | **Corregir** (deduplicar) | Si no, al unir, sus ventas se cuentan 2 veces |
| 2 | customers | email nulo y `not-an-email` | **Conservar** | No afecta las ventas; se reporta |
| 3 | products | `P088` precio -12.50, `P089` precio nulo | **Cuarentena** de sus líneas | No podemos inventar un precio |
| 4 | orders | `O00012` duplicado | **Corregir**: 1 versión por pedido (la de `updated_at` más reciente) | Evita duplicar importes |
| 5 | orders | estado `deliverd` | **Corregir** a `delivered` | Typo evidente, corrección segura |
| 6 | orders | `O00025` con fecha `2026-99-31` | **Cuarentena** | No se puede asignar a un mes |
| 7 | orders | `O00007` cliente `C9999` inexistente; `O00011` sin cliente | **Cuarentena** | Sin cliente no hay país |
| 8 | order_items | pedido `O99999` y producto `P999` inexistentes | **Cuarentena** | Referencias rotas |
| 9 | order_items | cantidad `0` y `-2` | **Cuarentena** | Fuera de rango |
| 10 | order_items | descuento `1.20` (120 %) | **Cuarentena** | Fuera de dominio, daría importe negativo |
| 11 | orders | pedidos `cancelled` | **Excluir** de ventas netas | Regla de negocio, no es un error |

> **Corregir** = arreglar con una regla segura. **Rechazar** = descartar. **Cuarentena** = apartar la fila,
> **guardada y con su motivo**, para que alguien la revise. El taller pide no descartar en silencio,
> así que usamos cuarentena con una columna `motivo_rechazo`.

## Paso 4: Limpiar cada tabla

Regla de oro: **limpia las tablas "dimensión" (clientes, productos, pedidos) ANTES de unirlas**,
sobre todo deja sus claves únicas. Así el join no multiplica filas.

In [8]:
# --- customers: quitar el duplicado exacto ---
# dropDuplicates(["customer_id"]) deja UNA fila por cada customer_id.
customers = customers_raw.dropDuplicates(["customer_id"])

# --- products: convertir el precio a número ---
products = (
    products_raw
    # decimal(10,2) = número exacto con 2 decimales (ideal para dinero; double tiene errores de redondeo)
    .withColumn("unit_price", F.expr("try_cast(unit_price AS decimal(10,2))"))
    # columna booleana: True si el precio sirve para calcular
    .withColumn("precio_valido", F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))
)

# --- orders: typo, fechas y una sola versión por pedido ---
# Ventana: "agrupa por order_id y, dentro de cada grupo, ordena del updated_at más nuevo al más viejo".
w_order = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

orders = (
    orders_raw
    # when(condición, valor).otherwise(valor) = el "if / else" de Spark
    .withColumn("status", F.when(F.col("status") == "deliverd", "delivered").otherwise(F.col("status")))
    .withColumn("order_ts",   F.expr("try_to_timestamp(order_ts)"))     # texto -> timestamp ('2026-99-31' -> null)
    .withColumn("updated_at", F.expr("try_to_timestamp(updated_at)"))
    .withColumn("rn", F.row_number().over(w_order))   # numera 1, 2, 3... dentro de cada order_id
    .filter("rn = 1")                                 # nos quedamos solo con la versión más reciente
    .drop("rn")                                       # la columna auxiliar ya no sirve
)

# --- order_items: convertir tipos ---
items = (
    items_raw
    .withColumn("quantity",     F.expr("try_cast(quantity AS int)"))
    .withColumn("discount_pct", F.expr("try_cast(discount_pct AS decimal(5,4))"))
)

print("customers:", customers_raw.count(), "->", customers.count())
print("orders:   ", orders_raw.count(),    "->", orders.count())

customers: 1001 -> 1000


orders:    20001 -> 20000


## Paso 5: Unir, clasificar cada línea y calcular el importe neto

1. Partimos de `items` (la tabla de **hechos**: cada fila es una línea vendida) y hacemos **LEFT joins**.
   *Left* significa "conserva TODAS las líneas, aunque no encuentren pareja". Así ninguna fila desaparece
   en silencio. Si no hay pareja, las columnas de la otra tabla quedan en `null`.
2. Creamos `motivo_rechazo`: una cadena de `when` que revisa los problemas en orden. La primera condición
   que se cumple pone el motivo. Si ninguna se cumple, queda `null` = **línea válida**.
3. Separamos válidas y cuarentena, y calculamos `net_amount` solo en las válidas.

In [9]:
detalle = (
    items
    .join(orders, "order_id", "left")                                             # + datos del pedido
    .join(customers.select("customer_id", "country"), "customer_id", "left")      # + país del cliente
    .join(products.select("product_id", "category", "unit_price", "precio_valido"),
          "product_id", "left")                                                   # + categoría y precio
)

motivo = (
    F.when(F.col("channel").isNull(), "pedido_huerfano")          # channel null => el order_id no existía en orders
    .when(F.col("category").isNull(), "producto_huerfano")        # category null => el product_id no existía
    .when(~F.col("precio_valido"), "precio_invalido")             # ~ significa NOT
    .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "cantidad_fuera_de_rango")   # | significa OR
    .when(F.col("discount_pct").isNull() | (F.col("discount_pct") < 0) | (F.col("discount_pct") >= 1),
          "descuento_fuera_de_dominio")
    .when(F.col("customer_id").isNull(), "pedido_sin_cliente")
    .when(F.col("country").isNull(), "cliente_huerfano")          # tiene customer_id pero no existe en customers
    .when(F.col("order_ts").isNull(), "fecha_pedido_invalida")
    .when(F.col("status") == "cancelled", "pedido_cancelado")     # regla de negocio
    # sin .otherwise(): si nada se cumple, el resultado es null = línea válida
)

# .cache() guarda el resultado en memoria: lo usaremos varias veces y no queremos recalcular los joins.
detalle = detalle.withColumn("motivo_rechazo", motivo).cache()

validas = (
    detalle
    .filter(F.col("motivo_rechazo").isNull())
    # importe neto = cantidad * precio * (1 - descuento), redondeado a 2 decimales
    .withColumn("net_amount",
                F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1) - F.col("discount_pct")), 2)
                .cast("decimal(18,2)"))                             # F.lit(1) = la constante 1 como columna
    .withColumn("mes", F.date_format("order_ts", "yyyy-MM"))       # '2026-01-08 22:54' -> '2026-01'
)
cuarentena = detalle.filter(F.col("motivo_rechazo").isNotNull())

print("¿A dónde fue cada línea?")
detalle.groupBy("motivo_rechazo").count().orderBy(F.desc("count")).show(truncate=False)

validas.select("line_id", "order_id", "quantity", "unit_price", "discount_pct", "net_amount", "mes").show(5)

¿A dónde fue cada línea?


+--------------------------+-----+
|motivo_rechazo            |count|
+--------------------------+-----+
|NULL                      |29434|
|pedido_cancelado          |9756 |
|precio_invalido           |823  |
|pedido_sin_cliente        |3    |
|fecha_pedido_invalida     |3    |
|cantidad_fuera_de_rango   |2    |
|cliente_huerfano          |2    |
|pedido_huerfano           |1    |
|producto_huerfano         |1    |
|descuento_fuera_de_dominio|1    |
+--------------------------+-----+

+--------+--------+--------+----------+------------+----------+-------+
| line_id|order_id|quantity|unit_price|discount_pct|net_amount|    mes|
+--------+--------+--------+----------+------------+----------+-------+
|L0000006|  O00003|       2|    585.05|      0.0500|   1111.60|2026-02|
|L0000007|  O00004|       3|    428.90|      0.0000|   1286.70|2026-01|
|L0000008|  O00004|       4|    151.30|      0.1500|    514.42|2026-01|
|L0000009|  O00005|       1|    186.00|      0.0000|    186.00|2026-02|
|L000

> **¿Por qué el orden de los `when` importa?** Una línea puede tener varios problemas, pero solo guardamos
> el **primero**. Ponemos primero los más graves (referencias rotas) y al final la regla de negocio (cancelado).

## Paso 6: Agregar ventas por mes, país, canal y categoría

Usamos **Spark SQL**: registramos el DataFrame como una "vista temporal" y le escribimos SQL normal.
Es exactamente lo mismo que usar `groupBy().agg()`; elige el estilo con el que te sientas más cómodo.

In [10]:
validas.createOrReplaceTempView("ventas_detalle")   # ahora 'ventas_detalle' existe como tabla para SQL

resumen = spark.sql("""
    SELECT mes, country, channel, category,
           COUNT(DISTINCT order_id) AS pedidos,      -- pedidos únicos (un pedido tiene varias líneas)
           COUNT(*)                AS lineas,
           SUM(net_amount)         AS ventas_netas
    FROM ventas_detalle
    GROUP BY mes, country, channel, category
    ORDER BY mes, country, channel, category
""")
resumen.show(10)
print("Filas del resumen:", resumen.count())

+-------+-------+-------+-----------+-------+------+------------+
|    mes|country|channel|   category|pedidos|lineas|ventas_netas|
+-------+-------+-------+-----------+-------+------+------------+
|2026-01|     CL| mobile|      books|    281|   339|   883541.09|
|2026-01|     CL| mobile|electronics|    266|   316|   744129.86|
|2026-01|     CL| mobile|       home|    279|   328|   756476.83|
|2026-01|     CL| mobile|     sports|    293|   345|   854666.82|
|2026-01|     CL|  store|      books|    261|   307|   763380.69|
|2026-01|     CL|  store|electronics|    251|   293|   655883.47|
|2026-01|     CL|  store|       home|    278|   327|   789399.51|
|2026-01|     CL|  store|     sports|    269|   327|   803753.78|
|2026-01|     CL|    web|      books|    273|   322|   858607.80|
|2026-01|     CL|    web|electronics|    283|   333|   755732.74|
+-------+-------+-------+-----------+-------+------+------------+
only showing top 10 rows

Filas del resumen: 144


## Paso 7: Guardar en Parquet y controles de calidad

**Parquet** es un formato de archivo por columnas, comprimido y con tipos. Es el estándar en data engineering
(mucho mejor que CSV para análisis). Spark escribe una **carpeta** con varios archivos `part-*.parquet` dentro.

`mode("overwrite")` = si la carpeta ya existe, la reemplaza. Esto es clave para la idempotencia.

In [11]:
resumen.write.mode("overwrite").parquet("output/ventas_mensuales")
validas.write.mode("overwrite").parquet("output/detalle_valido")
cuarentena.write.mode("overwrite").parquet("output/cuarentena")     # la cuarentena también se guarda: nada se pierde
print("Guardado en output/")

Guardado en output/


In [12]:
# Leemos de vuelta lo que guardamos: así probamos el archivo real, no el DataFrame en memoria.
resumen_guardado = spark.read.parquet("output/ventas_mensuales")

# QA1 - Conservación: cada línea leída está en válidas O en cuarentena. Ninguna se perdió ni se duplicó.
n_raw, n_val, n_cua = items_raw.count(), validas.count(), cuarentena.count()
print(f"QA1 conservación: {n_val} válidas + {n_cua} cuarentena = {n_val + n_cua} | leídas = {n_raw} ->",
      "OK" if n_val + n_cua == n_raw else "FALLA")

# QA2 - Reconciliación: la suma del detalle debe ser idéntica a la suma del resumen agregado.
total_detalle = validas.agg(F.sum("net_amount")).first()[0]          # .first()[0] saca el número del DataFrame
total_resumen = resumen_guardado.agg(F.sum("ventas_netas")).first()[0]
print(f"QA2 reconciliación: detalle = {total_detalle} | resumen = {total_resumen} ->",
      "OK" if total_detalle == total_resumen else "FALLA")

# QA3 - Sin duplicados por los joins: cada line_id debe aparecer una sola vez.
dups = validas.groupBy("line_id").count().filter("count > 1").count()
print(f"QA3 line_id únicos: {dups} duplicados ->", "OK" if dups == 0 else "FALLA")

QA1 conservación: 29434 válidas + 10592 cuarentena = 40026 | leídas = 40026 -> OK


QA2 reconciliación: detalle = 71680923.28 | resumen = 71680923.28 -> OK


QA3 line_id únicos: 0 duplicados -> OK


## Paso 8: Carga incremental idempotente

Llega un lote nuevo (JSON Lines: un objeto JSON por línea) con **100 pedidos nuevos** y **una actualización
tardía** del pedido histórico `O00005` (mismo pedido, `updated_at` más nuevo).

**Idempotente** = ejecutar el proceso 1 vez o 10 veces con el mismo lote da **el mismo resultado**.

Estrategia:
1. Unir histórico + lote (`unionByName`).
2. Por cada `order_id`, quedarse solo con la versión de `updated_at` más reciente (`row_number`).
3. Para las líneas, deduplicar por `line_id`.
4. Escribir con `overwrite`, nunca con `append` a ciegas (append duplicaría si reprocesas).

In [13]:
orders_inc = spark.read.json("datos/incremental/orders_incremental.jsonl")
items_inc  = spark.read.json("datos/incremental/order_items_incremental.jsonl")
print("Lote: pedidos =", orders_inc.count(), "| líneas =", items_inc.count())

# ¿Qué pedido del lote ya existía en el histórico? Esa es la actualización tardía.
orders_inc.join(orders_raw.select("order_id"), "order_id", "left_semi").show(truncate=False)
# left_semi = "filas de la izquierda que SÍ tienen pareja" (lo contrario de left_anti)

Lote: pedidos = 101 | líneas = 200
+--------+-------+-----------+--------------------+---------+--------------------+
|order_id|channel|customer_id|order_ts            |status   |updated_at          |
+--------+-------+-----------+--------------------+---------+--------------------+
|O00005  |store  |C0461      |2026-02-22T15:20:00Z|delivered|2026-03-05T12:00:00Z|
+--------+-------+-----------+--------------------+---------+--------------------+



In [14]:
def aplicar_lote(orders_base, items_base, orders_lote, items_lote):
    """Combina base + lote y deja la versión más reciente de cada clave."""
    # select(columnas de la base): el JSON puede traer las columnas en otro orden; así quedan alineadas.
    o = (
        orders_base.unionByName(orders_lote.select(orders_base.columns))   # unionByName une por NOMBRE de columna
        .withColumn("rn", F.row_number().over(
            Window.partitionBy("order_id")
                  .orderBy(F.expr("try_to_timestamp(updated_at)").desc())))  # la más nueva queda con rn = 1
        .filter("rn = 1").drop("rn")
    )
    i = (
        items_base.unionByName(items_lote.select(items_base.columns))
        .dropDuplicates(["line_id"])                                       # una fila por línea
    )
    return o, i

# Aplicamos el lote UNA vez...
o1, i1 = aplicar_lote(orders_raw, items_raw, orders_inc, items_inc)
# ...y otra vez el MISMO lote sobre el resultado (simula un reproceso por error).
o2, i2 = aplicar_lote(o1, i1, orders_inc, items_inc)

print("Pedidos tras 1 vez:", o1.count(), "| tras 2 veces:", o2.count())
print("Líneas  tras 1 vez:", i1.count(), "| tras 2 veces:", i2.count())

print("El pedido O00005 quedó con la versión más reciente:")
o2.filter("order_id = 'O00005'").show(truncate=False)

Pedidos tras 1 vez: 20100 | tras 2 veces: 20100


Líneas  tras 1 vez: 40226 | tras 2 veces: 40226
El pedido O00005 quedó con la versión más reciente:
+--------+-----------+--------------------+---------+-------+--------------------+
|order_id|customer_id|order_ts            |status   |channel|updated_at          |
+--------+-----------+--------------------+---------+-------+--------------------+
|O00005  |C0461      |2026-02-22T15:20:00Z|delivered|store  |2026-03-05T12:00:00Z|
+--------+-----------+--------------------+---------+-------+--------------------+



**Resultado:** 20.100 pedidos y 40.226 líneas tanto con 1 como con 2 aplicaciones → **idempotente**.
`O00005` quedó con `updated_at = 2026-03-05` → **la actualización más reciente tiene precedencia**.

> Con Delta Lake esto se haría con un `MERGE INTO`, pero el taller dice que no es necesario.
> Para recalcular las ventas con el lote incluido, bastaría con repetir los pasos 4 a 7 usando `o1` e `i1`
> en lugar de `orders_raw` e `items_raw`.


In [15]:
spark.stop()   # libera la JVM y la memoria al terminar